# How the News Pyramid Broke: A Bayesian Analysis of Truth, Style, and Attention
**Ananya Sen**

**Central question.** Do articles that use more attention-grabbing language actually get
more engagement, and are they more likely to be misinformation?

**Approach.**
(1) characterize the stylistic signature of misinformation with a Bayesian logistic regression on 21,724 fact-checker-labeled headlines (FakeNewsNet)
(2) estimate the association between attention-optimized language and engagement with a hierarchical Bayesian model that partially pools slopes across collection strata
(3) estimate the *causal* effect of headline style on clicks using 2,607 randomized Upworthy headline A/B tests (12,010 arms). A generative–discriminative pair experiment validates the feature pipeline. All estimates are reported as posteriors with credible intervals.

## 0. Setup

Run this once, then **restart the kernel** before continuing (fresh installs are often not
picked up by a running kernel).

In [1]:
%pip install -q -U pymc arviz vaderSentiment scikit-learn joblib h5netcdf pyarrow

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sklearn-compat 0.1.5 requires scikit-learn<1.9,>=1.2, but you have scikit-learn 1.9.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [1]:
import sys
%cd "/Users/ananyasen/Bayesian ML/Final Project/attentionlens_final"

/Users/ananyasen/Bayesian ML/Final Project/attentionlens_final


## 1. Data

**FakeNewsNet** (Shu et al.; released CSVs from github.com/KaiDMML/FakeNewsNet): 21,724
deduplicated headlines from the PolitiFact and GossipCop subsets with fact-checker
veracity labels. Engagement proxy = number of tweets sharing each article (tweet-ID counts
in the release; no Twitter API required). *Limitation:* tweet counts reflect FakeNewsNet's
collection methodology, which differs across the four source x label strata — median counts
range from 6 (politifact_real) to 79 (politifact_fake) — so cross-strata engagement
comparisons are unreliable; our engagement model therefore identifies only within-stratum
variation.

**Upworthy Research Archive** (Matias et al., osf.io/jd64p, exploratory packages release):
randomized headline A/B tests — same article, same period, randomly assigned viewers,
differing only in headline — with recorded impressions and clicks. Within-test comparisons
eliminate confounding by topic, timing, and audience by design, licensing causal language.
*Limitations:* one progressive publisher, 2013–2015, clicks rather than shares.

## 2. Feature extraction

Attention-language features per headline: sensational-word ratio, urgency terms,
capitalization ratios, exclamations/questions, clickbait openers, second-person address,
number presence, VADER sentiment (positive/negative/intensity), and headline length.

In [2]:
!{sys.executable} features.py

21724 articles, 24.5% fake
       sensational_ratio  urgency_ratio  caps_ratio
count         21724.0000     21724.0000  21724.0000
mean              0.0036         0.0041      0.1614
std               0.0199         0.0205      0.0759
min               0.0000         0.0000      0.0000
25%               0.0000         0.0000      0.1250
50%               0.0000         0.0000      0.1667
75%               0.0000         0.0000      0.1957
max               0.5000         0.5000      1.0000


## 3. Design decisions

1. **Circularity guard.** The attention score (a supervised model mapping language features
   to engagement) is trained on split A; every hypothesis test runs on held-out split B.
   Otherwise "a score trained on engagement predicts engagement" would be true by construction.
2. **Headline length is excluded from the attention score** and enters all models as a
   control instead. Length dominated the score's fit but is a length artifact, not attention
   *language*; with it included, a neutral procedural headline scored identically to clickbait.
3. **Hierarchical partial pooling across collection strata.** Because the engagement proxy's
   scale differs across the four source x label strata for methodological reasons, the
   engagement slope is estimated within strata and pooled partially (non-centered
   parameterization) rather than trusting pooled cross-strata contrasts.
4. **Raw style features as the primary misinformation model.** Regressing the fake label on
   the engagement-trained score yields a large negative coefficient — the opposite sign of
   the raw-feature model — which we retain as an appendix: the divergence is evidence that
   the engagement proxy partially encodes collection process rather than organic virality.
5. **Synthetic pairs are validation/demo only** — never classifier training data (a model
   trained on generated sensationalism would learn generator style, not misinformation).

## 4. Appendix analysis — attention score + score-based models (split A/B)

Trains the attention score and fits the original score-based engagement and misinformation
models. Kept as the appendix demonstrating the proxy-contamination sign flip discussed above.

In [3]:
!{sys.executable} models.py

Attention-score model: top language features (ridge coefs, std'ized):
questions             -0.068
vader_pos              0.065
emotional_intensity   -0.062
vader_neg              0.050
exclamations          -0.041
has_number             0.034
urgency_ratio          0.034
clickbait_opener       0.031 

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [alpha, b_attention, b_gossipcop, b_title_len, sigma]
Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 6 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [alpha, b_attention, b_gossipcop, b_title_len]
Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 13 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics

H1: log-engagement ~ 

## 5. Primary Bayesian analyses (FakeNewsNet, held-out split)

**H1:** hierarchical model of log-engagement on the attention score, slopes partially pooled
across the four collection strata. **H2:** Bayesian logistic regression of the fake label on
raw style features. Final sampling settings: 4 chains x 2000 draws, target_accept = 0.95.

In [4]:
!{sys.executable} models_v2.py

Initializing NUTS using jitter+adapt_diag...
Sequential sampling (4 chains in 1 job)
NUTS: [mu_alpha, sd_alpha, z_alpha, mu_b_attention, sd_b_attention, z_b, b_title_len, sigma]
Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 854 seconds.
There were 29 divergences after tuning. Increase `target_accept` or reparameterize.
H1 hierarchical: attention-score slope, pooled + per stratum
                                 mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  ess_tail  r_hat
mu_b_attention                  0.154  0.191  -0.237    0.509      0.004    0.005    3444.0    3063.0    1.0
b_attention_s[gossipcop_fake]   0.291  0.103   0.101    0.482      0.001    0.001    5947.0    6428.0    1.0
b_attention_s[gossipcop_real]   0.146  0.060   0.030    0.256      0.001    0.001    7759.0    5782.0    1.0
b_attention_s[politifact_fake]  0.092  0.221  -0.371    0.480      0.003    0.003    7137.0    6231.0    1.0
b_attention_s[politifact_re

## 6. Causal analysis — Upworthy randomized headline tests

Outcome: empirical log-odds of clicking, centered within test; features centered within test
(fixed-effects equivalent); arms weighted by impressions. All 2,607 qualifying tests in the
exploratory release are used (no subsampling).

In [5]:
!{sys.executable} upworthy_h1.py --csv "../upworthy-archive-exploratory-packages-03.12.2020.csv" --draws 2000

2607 tests, 12010 arms
Initializing NUTS using jitter+adapt_diag...
Sequential sampling (4 chains in 1 job)
NUTS: [beta, sigma]
/opt/anaconda3/lib/python3.11/site-packages/pymc/step_methods/hmc/quadpotential.py:316: RuntimeWarning: overflow encountered in dot
  return 0.5 * np.dot(x, v_out)
Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 25 seconds.

Causal effect of style on within-test log-odds of clicking
(standardized; positive = that feature attracts clicks):
                      mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  ess_tail  r_hat
questions           -0.031  0.003  -0.036   -0.026        0.0      0.0   15794.0    6378.0    1.0
title_len_words      0.028  0.003   0.022    0.034        0.0      0.0   10297.0    6354.0    1.0
vader_neg            0.017  0.003   0.011    0.023        0.0      0.0   10368.0    7417.0    1.0
has_number           0.016  0.003   0.011    0.021        0.0      0.0   14469.0    6474.0    1

## 7. Generative–discriminative pair experiment

For each headline we generate a style-matched counterfactual (neutral vs sensationalized,
same content) and test whether the pipeline separates the pair. Rule mode below is a
plumbing check only (the generator shares the feature lexicon, so near-perfect separation is
expected). The meaningful run uses an LLM generator whose mechanism is independent of the
feature set:

```
export ANTHROPIC_API_KEY=...   # then:
python pairs.py --mode anthropic --n 200
```

In [6]:
!{sys.executable} pairs.py --mode rule --n 300

[mode=rule] Sanity check only — generator shares the feature lexicon; near-perfect separation is expected, not evidence.

n pairs: 300
STYLE discriminator (which version is sensationalized?):
  accuracy: 99.7%   mean gap +6.58
ENGAGEMENT-trained attention score (does sensationalizing raise predicted engagement?):
  sensational scored higher in 15.7% of pairs
  mean within-pair score gap: -0.347  [89% interval -0.377, -0.316]
Saved pairs_rule.parquet


## 8. Results

In the FakeNewsNet corpus (n = 21,724 headlines), the Bayesian logistic regression finds fake-labeled headlines are credibly more interrogative (questions: +0.29 standardized log-odds, 94% HDI [0.25, 0.33]), more negative in sentiment (+0.19 [0.13, 0.25]), higher in capitalization (+0.14 [0.08, 0.21]), and more sensational in vocabulary (+0.09 [0.05, 0.13]); classic celebrity-clickbait markers (second person: −0.18 [−0.25, −0.12]; "this/why" openers: −0.09 [−0.13, −0.04]; emotional intensity: −0.13 [−0.20, −0.05]) skew real in this corpus because GossipCop's real class is entertainment news — misinformation style is specifically negative/alarmist, not engagement-bait. The hierarchical engagement model finds only a small within-stratum association between attention-optimized language and engagement (pooled μ = 0.15 [−0.24, 0.51]; largest stratum slope: gossipcop_fake 0.29 [0.10, 0.48]), with headline length a stronger predictor (−0.19 [−0.22, −0.16]) and cross-strata comparisons unreliable for the collection-methodology reasons described above. In 2,607 randomized Upworthy tests (12,010 headline arms), negative sentiment causally increases clicks (+0.017 [0.011, 0.023]) and sensational vocabulary adds a marginal premium (+0.010 [0.005, 0.015]), while question marks (−0.031 [−0.036, −0.026]), exclamations (−0.014 [−0.019, −0.009]), and capitalization (−0.009 [−0.016, −0.002]) reduce them.

**Together: misinformation has a distinctive stylistic signature, but of its markers only negativity — and, marginally, sensational vocabulary — is causally rewarded with attention; the strongest misinformation marker (question-style headlines) actually reduces clicks. The "attention premium" of misinformation style is small and largely limited to negative emotional tone.**

**Sampling diagnostics:** All models sampled with NUTS, 4 chains × 2,000 draws after 2,000 tuning iterations. R̂ = 1.00 for all reported parameters, with bulk effective sample sizes between ~3,400 and ~16,400. The hierarchical engagement model (non-centered parameterization, target_accept = 0.95) recorded 29 divergent transitions out of 8,000 post-warmup draws (0.4%); estimates were stable across preliminary and final runs, and all other models sampled without divergences.

## 9. Limitations and ongoing work

**Limitations.** Headlines only (no article bodies); engagement proxy validity (tweet
collection methodology, discussed above); Upworthy external validity (one publisher,
2013–2015, clicks not shares); AllSides outlet-level bias labels inherit to all of an
outlet's articles.

**Next steps.** LLM-mode pair experiment; AllSides outlet-leaning moderation of the
style–misinformation relationship (hierarchical by leaning); transformer emotion features
(go_emotions); AttentionLens dashboard demo (attention score + P(misinformation) with
credible intervals + news-pyramid reconstruction); GDELT longitudinal trend analysis as
ongoing fellowship work on how the news pyramid has changed in the attention economy.